# Observing System Design

Everything so far used *all* 11 242 ITP samples. An observing system experiment asks: **what if we had different
observations?** Fewer ITPs, ITPs somewhere else, a mooring instead of a drifter, only summer data. Each such observing
system trains its own network, each network reconstructs the Arctic, and the reconstruction skills are compared —
the two-network comparison of the [landing page](intro.md), generalised.

The ingredients are already there. What is new is a way of **selecting subsets** of the samples:

| selector | meaning |
|---|---|
| `selectors.everything()` | all samples (the reference) |
| `selectors.tracks(95, 5)` | the ITPs with these ids |
| `selectors.region(lon=(-180, -120), lat=(70, 80))` | a box (longitude goes east from the first to the second value) |
| `selectors.near(lat=77, lon=-138, radius_km=75)` | all samples within a radius of a point: a pseudo-mooring |
| `selectors.years(2008, 2010)`, `.months(6, 7, 8)`, `.season("JJA")` | time windows |
| `selectors.every(5)` | every 5th sample: a thinned observing system |

Selectors are boolean masks and combine with `&`, `|`, `~`. `selectors.describe(mask)` says what a selection contains.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

In [ ]:
beaufort_summer = selectors.region(lon=(-180, -120)) & selectors.season("JJA")
print("Beaufort Sea, summer:", selectors.describe(beaufort_summer))
print("ITP 95 or ITP 5:     ", selectors.describe(selectors.tracks(95, 5)))

The track ids, for picking ITPs:

In [ ]:
plots.tracks(min_days=30, figsize=(8, 8));

## Running an experiment

`experiments.run_experiment(name, scenarios)` takes a dictionary *label → selection*, adds the `all` reference, and
for each observing system trains the network (three times, with different random seeds, to see the training noise),
reconstructs the Arctic and scores it. It then draws three things:

1. **where** each observing system sampled,
2. the **reconstruction skill** of each (bar = mean over seeds, whisker = min–max, number = samples),
3. **maps of skill lost**: the skill of the `all` reference minus the skill of the scenario — red where the missing
   observations were helping.

The networks are trained for 15 epochs here instead of 30 (the curve is flat by then), so one observing system takes
about ten seconds per seed.

## Experiment A — *where*: one ITP at a time

Three ITPs that drifted through different parts of the Arctic, each used alone, plus the pair from the landing page.

In [ ]:
results_A = experiments.run_experiment("A: where", {
    "ITP 95":     selectors.tracks(95),       # two months north of the Laptev Sea
    "ITP 5":      selectors.tracks(5),        # six months drifting toward Fram Strait
    "ITP 15":     selectors.tracks(15),       # six months north of the Canadian Archipelago
    "ITP 95 + 5": selectors.tracks(95, 5),    # NN 1 -> NN 2 of the landing page
})

Where a single ITP was matters as much as how long it lasted: ITP 95 has a third of the samples of ITP 5 and
reaches about the same skill. Putting the two together (NN 1 → NN 2) lifts the skill well above either — two
regions beat one — but it is still short of the full set. The difference maps say where each network falls short:
far from its own track, as expected, with the Beaufort Gyre (where most of the full set's data are) lighting up for
all of them.

## Experiment B — *how many*: adding ITPs one at a time

Start from one long ITP track in the Beaufort Sea and add one track at a time: first the other long Beaufort
tracks, then — following a simple design rule — always the long track that is *farthest from everything we already
observe*. The dashed line is the skill with all samples.

In [ ]:
sequence = experiments.farthest_first_sequence()        # [1, 2, 3, 4, 96, 24, 7, 27, 5, 15]
results_B = experiments.run_sequence("B: how many", sequence, map_steps=(1, 4, 8, 10))

Two things to read off the curve (the axis is zoomed; the changes are larger than the seed spread). First, the three
extra Beaufort tracks (steps 2–4, +1 500 samples) buy almost nothing, while the first tracks from *other* regions
(steps 5–7, +1 000 samples) lift the skill to the all-ITP line: **a new region is worth more than more data from a
region we already observe**. Second, six or seven well-placed ITPs — about a quarter of the samples — do as well as all
105 tracks, and adding more changes nothing. That is one answer to "what is the minimum observing system?" for this
problem — with the caveat that the dashed line is not an upper bound: the full set is itself concentrated in the
Beaufort Gyre, and the maps show the regions no ITP ever reached staying red at every step.

## Design your own experiment

An experiment is a dictionary: each entry is a **label** (any text — it becomes the bar label and map title) and a
**selection** built from the selectors above. To make your own:

1. **Choose a question** — *where* (regions, single ITPs), *how many* (thinning, sequences), *when* (seasons, years),
   *what kind* (a mooring: `near`; a drifter: `tracks`).
2. **Write one line per observing system** in the dictionary and check each with `selectors.describe` — an empty
   or tiny selection (a few dozen samples) trains a poor network and tells you little.
3. **Run it.** `quick=True` trains one network for 10 epochs per scenario for a fast look (the seed spread is then
   unknown); drop it for the full three seeds. For a growing sequence of ITPs use `experiments.run_sequence`.

The template below compares a pseudo-mooring with a single long drifter, a seasonal subset and a latitude band.

In [ ]:
my_experiment = {
    "mooring, 9 yr":    selectors.near(lat=77.1, lon=-138, radius_km=75),       # all ITP days within 75 km of one spot
    "one buoy, 2 yr":   selectors.tracks(22),                                   # the longest single ITP track
    "Beaufort, summer": selectors.region(lon=(-180, -120)) & selectors.season("JJA"),
    "north of 85 N":    selectors.region(lat=(85, 90)),
}
for label, sel in my_experiment.items():
    print(f"{label:18s} {selectors.describe(sel)}")

my_results = experiments.run_experiment("my experiment", my_experiment, quick=True)

Some questions to try, each one line in the dictionary:

* **Thinning.** `selectors.every(5)` against `everything()`: does an observing system with a fifth of the samples lose anything?
  Where does the loss appear first?
* **Mooring vs drifter.** The template's first two lines, with the radius or the track changed. Is a long record at one
  point worth more than a shorter record that moves?
* **Season.** `selectors.season("DJF")` vs `"JJA"`: is winter data or summer data more useful for reconstructing a
  November–December period? (The maps are from late autumn — does training on the matching season help?)
* **Order matters.** In experiment B, swap the order of addition: `experiments.run_sequence("B'", [1, 2, 3, 4, 7, 27, 24, 96, 5, 15])`
  and compare the curves. The *same* ten ITPs give a different path to the same end point.
* **Your region.** `selectors.region(lon=(0, 90), lat=(80, 90))` — the Eurasian Basin: how much skill do a few hundred
  samples there add to the Beaufort-heavy full set? (Combine with `|`.)

```{admonition} The real question
:class: note
Everything here is a *testbed*: the "truth" is a climate model, the observations are synthetic, the maps are ten days
long. The method is the point — the same workflow with real observations and a reanalysis as the input fields is how
Observing System Design is done for the real Arctic, and the same network, trained on real buoys, is a real
reconstruction. The repository's single notebook, [`reconstruction.ipynb`](https://github.com/lahoffman/ml-tutorial-OAISIS/blob/main/reconstruction.ipynb),
has every line of code behind these pages in one place.
```